# Book-NCF: Exploratory Data Analysis (EDA)

**Notebook Objectives:**
1. **Dataset Overview**: Inspect raw structures of `ratings.csv` and `books.csv`.
2. **Interaction Matrix Sparsity**: Quantify data sparsity across the user-item interaction space.
3. **Long-Tail Distribution**: Analyze user activity levels and book popularity skews.
4. **Rating Score Distribution**: Examine rating values and justify binarization for implicit feedback modeling.

In [ ]:
import os
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

# Read data paths from config.yaml with fallback to data/raw/
try:
    with open("../configs/config.yaml", "r") as f:
        cfg = yaml.safe_load(f)
    RATINGS_PATH = cfg["data"]["ratings_path"]
    BOOKS_PATH = cfg["data"]["books_path"]
except Exception:
    RATINGS_PATH = "../data/raw/ratings.csv"
    BOOKS_PATH = "../data/raw/books.csv"

## 1. Load and Inspect Raw Datasets

In [ ]:
if os.path.exists(RATINGS_PATH):
    ratings_df = pd.read_csv(RATINGS_PATH)
    print(f"Ratings shape: {ratings_df.shape}")
    display(ratings_df.head())
else:
    print(f"{RATINGS_PATH} not found. Please place ratings.csv in data/raw/")

In [ ]:
if os.path.exists(BOOKS_PATH):
    books_df = pd.read_csv(BOOKS_PATH)
    print(f"Books shape: {books_df.shape}")
    display(books_df.head())
else:
    print(f"{BOOKS_PATH} not found. Please place books.csv in data/raw/")

## 2. Interaction Matrix Sparsity Calculation

In [ ]:
if 'ratings_df' in locals():
    user_col = "user_id" if "user_id" in ratings_df.columns else ratings_df.columns[0]
    book_col = "book_id" if "book_id" in ratings_df.columns else ratings_df.columns[1]

    n_users = ratings_df[user_col].nunique()
    n_books = ratings_df[book_col].nunique()
    n_interactions = len(ratings_df)
    
    total_possible = n_users * n_books
    sparsity = (1.0 - (n_interactions / total_possible)) * 100.0
    
    print(f"Total Unique Users: {n_users:,}")
    print(f"Total Unique Books: {n_books:,}")
    print(f"Total Interactions: {n_interactions:,}")
    print(f"Interaction Matrix Sparsity: {sparsity:.4f}%")

## 3. Long-Tail Distributions: User Activity & Item Popularity

In [ ]:
if 'ratings_df' in locals():
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    # User Activity Distribution
    user_counts = ratings_df[user_col].value_counts()
    sns.histplot(user_counts, bins=50, kde=True, ax=axes[0], color="royalblue", log_scale=(False, True))
    axes[0].set_title("User Activity Distribution (Log Scale)")
    axes[0].set_xlabel("Number of Ratings per User")
    axes[0].set_ylabel("Count (Log Scale)")
    
    # Book Popularity Distribution
    book_counts = ratings_df[book_col].value_counts()
    sns.histplot(book_counts, bins=50, kde=True, ax=axes[1], color="crimson", log_scale=(False, True))
    axes[1].set_title("Book Popularity Distribution (Log Scale)")
    axes[1].set_xlabel("Number of Ratings per Book")
    axes[1].set_ylabel("Count (Log Scale)")
    
    plt.tight_layout()
    plt.show()

## 4. Rating Score Distribution & Implicit Feedback Conversion

In [ ]:
if 'ratings_df' in locals() and 'rating' in ratings_df.columns:
    plt.figure(figsize=(8, 4))
    sns.countplot(data=ratings_df, x="rating", palette="viridis")
    plt.title("Original Rating Distribution")
    plt.xlabel("Rating Score")
    plt.ylabel("Frequency")
    plt.show()
    
    print("Note: In NCF, interactions are treated as implicit feedback,")
    print("where observed interactions are labeled 1 (positive) and unobserved items are sampled as 0 (negative).")